<a href="https://colab.research.google.com/github/michael-palomino-tm/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/blob/Mistral/1B_basic_planning_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1B. Planificación básica de agentes: del plan a la ejecución

**ISY0101 — Ingeniería de Soluciones con IA | Actividad 2.3.1**

### Objetivos de aprendizaje
1. Distinguir planificación de ejecución.
2. Generar un plan **estructurado** mediante un LLM, restringido a herramientas reales.
3. Ejecutar tareas en orden, **compartiendo el estado** entre pasos.
4. Observar fallas y explicar por qué este ejemplo **todavía no es una orquestación multiagente**.

**Problema:** investigar una empresa usando Wikipedia, identificar a su director ejecutivo a partir de datos estructurados de Wikidata y buscar su biografía.

> En este laboratorio hay un planificador LLM y un ejecutor de Python. No se crean varios agentes autónomos. La herramienta Wikipedia puede no estar disponible o contener información desactualizada; sus resultados se deben verificar antes de tomar decisiones reales.

## 1. Instalación y configuración

Conservamos la convención del notebook original: **Google Colab → Secrets** (🔑), **local → `.env`**. No se escriben claves en el código ni se imprimen.

Secret obligatorio: `LLM_API_KEY`. Opcionales: `LLM_BASE_URL`, `LLM_MODEL`, `LLM_MODEL_SMALL`, `LANGSMITH_API_KEY`. El modelo por defecto usa la API compatible con OpenAI de Mistral.

In [1]:
# Instalación solo en Google Colab
import sys
if "google.colab" in sys.modules:
    %pip install -q langchain-openai python-dotenv requests
# En local: pip install langchain-openai python-dotenv requests


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.4/146.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.3/573.3 kB 10.2 MB/s eta 0:00:00


In [2]:
import os

try:
    from google.colab import userdata
    for key in ("LLM_API_KEY", "LANGSMITH_API_KEY", "LLM_BASE_URL", "LLM_MODEL", "LLM_MODEL_SMALL"):
        try:
            value = userdata.get(key)
            if value:
                os.environ[key] = value
        except Exception:
            pass
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

os.environ.setdefault("LLM_BASE_URL", "https://api.mistral.ai/v1")
os.environ.setdefault("LLM_MODEL", "mistral-small-latest")
os.environ.setdefault("LLM_MODEL_SMALL", "ministral-8b-latest")

if not os.getenv("LLM_API_KEY"):
    raise RuntimeError("Falta LLM_API_KEY. Configúralo en Colab Secrets o en tu archivo .env.")

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model=os.environ["LLM_MODEL"],
    base_url=os.environ["LLM_BASE_URL"],
    api_key=os.environ["LLM_API_KEY"],
    temperature=0,
    timeout=45,
    max_retries=2,
)
print("LLM configurado:", os.environ["LLM_MODEL"])
print("Credenciales cargadas (valor oculto).")

LLM configurado: ministral-14b-latest
Credenciales cargadas (valor oculto).


## 2. Contrato de herramientas y estado

El **ejecutor** solo permite tres operaciones: `wiki_summary` (consultar Wikipedia), `wikidata_ceo` (consultar el CEO mediante datos estructurados de Wikidata) y `compose_report` (redactar un resumen apoyado en los resultados).

El planificador **no puede inventar herramientas**. Las dependencias se resuelven con variables de estado (`company`, `company_summary`, `ceo`, `bio_summary`).

In [3]:
import json
import re
import requests
from pprint import pprint

ALLOWED_TOOLS = {"wiki_summary", "wikidata_ceo", "compose_report"}

# Datos de entrada. El modo alternativo es explícito y se informa en pantalla.
company = "NVIDIA"
goal = f"Investiga la empresa {company}, identifica a su CEO a partir de la consulta y resume su biografía en Wikipedia."
state = {"company": company}

# Si Wikipedia falla, usa textos preparados SOLO para demostrar el flujo.
# Estos textos NO deben considerarse una consulta real ni información actualizada.
DEMO_FALLBACK = True
DEMO_TEXTS = {
    "NVIDIA": (
        "TEXTO DIDÁCTICO (no obtenido de Wikipedia en tiempo real): "
        "NVIDIA es una empresa tecnológica que desarrolla hardware y software de computación. "
        "Su director ejecutivo (CEO) es Jensen Huang."
    ),
    "Jensen Huang": (
        "TEXTO DIDÁCTICO (no obtenido de Wikipedia en tiempo real): "
        "Jensen Huang es un empresario y cofundador de NVIDIA. "
        "Ha desempeñado el cargo de director ejecutivo de la compañía."
    ),
}
used_demo_data = []
print("Objetivo:", goal)
print("Fallback didáctico activado:", DEMO_FALLBACK)


Objetivo: Investiga la empresa NVIDIA, identifica a su CEO a partir de la consulta y resume su biografía en Wikipedia.
Fallback didáctico activado: True


## 3. Planificador: un LLM genera un plan, NO ejecuta acciones

Solicitamos pasos en formato JSON para evitar interpretar instrucciones libres mediante cortes de cadenas. El plan se valida antes de ejecutarse. El validador acepta el nombre literal de la empresa en el primer paso y lo normaliza a una referencia de estado.

**Nota:** para concentrarnos en el principio de planificación, las operaciones autorizadas y sus dependencias están fijadas. El LLM devuelve un plan dentro de esas reglas; este ejercicio no demuestra planificación abierta ni generación libre de estrategias.


In [4]:
from langchain_core.prompts import ChatPromptTemplate

planner_prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres un planificador de tareas para un ejecutor MUY LIMITADO.
Solo existen estas herramientas:
- wiki_summary: busca en Wikipedia un tema indicado por 'query'.
- wikidata_ceo: consulta el CEO de una empresa en Wikidata usando el nombre de la empresa.
- compose_report: crea el informe usando el estado existente.
Devuelve ÚNICAMENTE un objeto JSON válido con clave 'steps' (una lista).
Cada paso tiene: id (entero), tool, input (cadena), output (cadena).
El plan DEBE contener exactamente estos 4 pasos, en este orden:
1 wiki_summary, input '{{{{company}}}}', output 'company_summary'
2 wikidata_ceo, input '{{{{company}}}}', output 'ceo'
3 wiki_summary, input '{{{{ceo}}}}', output 'bio_summary'
4 compose_report, input 'state', output 'report'
No inventes nombres de CEO ni herramientas; la identidad se obtiene durante la ejecución.
No agregues markdown ni explicaciones."""),
    ("human", "Objetivo: {goal}")
])

raw_plan = (planner_prompt | llm).invoke({"goal": goal}).content
print("Respuesta del planificador:")
print(raw_plan)

Respuesta del planificador:
```json
{
  "steps": [
    {
      "id": 1,
      "tool": "wiki_summary",
      "input": "NVIDIA",
      "output": "company_summary"
    },
    {
      "id": 2,
      "tool": "wikidata_ceo",
      "input": "NVIDIA",
      "output": "ceo"
    },
    {
      "id": 3,
      "tool": "wiki_summary",
      "input": "{{ceo}}",
      "output": "bio_summary"
    },
    {
      "id": 4,
      "tool": "compose_report",
      "input": "state",
      "output": "report"
    }
  ]
}
```


In [6]:
def parse_json_object(text):
    # Algunos proveedores agregan fences aun cuando se pide JSON.
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.I)
    return json.loads(cleaned)

def validate_plan(plan):
    expected = [
        (1, "wiki_summary", "{{company}}", "company_summary"),
        (2, "wikidata_ceo", "{{company}}", "ceo"),
        (3, "wiki_summary", "{{ceo}}", "bio_summary"),
        (4, "compose_report", "state", "report"),
    ]
    steps = plan.get("steps")
    if not isinstance(steps, list) or len(steps) != len(expected):
        raise ValueError("Plan inválido: se requieren exactamente cuatro pasos.")
    for step, (sid, tool, inp, out) in zip(steps, expected):
        if not isinstance(step, dict):
            raise ValueError(f"El paso debe ser un objeto: {step}")

        # En el primer paso, el LLM puede escribir el nombre literal
        # de la empresa en vez de la referencia al estado {{company}}.
        # Ambas opciones son equivalentes para nuestro ejecutor.
        if sid in (1, 2) and step.get("input") == company:
          step["input"] = "{{company}}"

        actual = (step.get("id"), step.get("tool"), step.get("input"), step.get("output"))
        if actual != (sid, tool, inp, out):
            raise ValueError(f"Paso incompatible con las herramientas/dependencias: {step}")
        if tool not in ALLOWED_TOOLS:
            raise ValueError(f"Herramienta desconocida: {tool}")
    return steps

plan = parse_json_object(raw_plan)
steps = validate_plan(plan)
print("Plan válido. Pasos:")
pprint(steps)

Plan válido. Pasos:
[{'id': 1,
  'input': '{{company}}',
  'output': 'company_summary',
  'tool': 'wiki_summary'},
 {'id': 2, 'input': '{{company}}', 'output': 'ceo', 'tool': 'wikidata_ceo'},
 {'id': 3, 'input': '{{ceo}}', 'output': 'bio_summary', 'tool': 'wiki_summary'},
 {'id': 4, 'input': 'state', 'output': 'report', 'tool': 'compose_report'}]


## 4. Ejecutor: operaciones y transferencia de contexto

Se consulta Wikipedia por HTTP para el resumen y Wikidata para localizar la propiedad estructurada **P169 (chief executive officer)** de la empresa. La segunda consulta no depende de que el nombre figure en la introducción de Wikipedia.
Si la API no responde, el modo `DEMO_FALLBACK=True` permite continuar con **textos didácticos locales** para NVIDIA y Jensen Huang. En tal caso el notebook muestra una advertencia: **no se está consultando Wikipedia y los datos no están verificados ni actualizados**.

Para exigir únicamente información en tiempo real, cambia `DEMO_FALLBACK = False`. Este modo permite explicar también la diferencia entre **fallos de infraestructura** y **limitaciones del plan estático**.


In [7]:
def resolve_input(value, state):
    match = re.fullmatch(r"\{\{([a-z_]+)\}\}", value)
    if match:
        key = match.group(1)
        if not state.get(key):
            raise ValueError(f"No existe un resultado utilizable para '{key}'.")
        return state[key]
    return value

def wikipedia_summary(query):
    """Consulta Wikipedia mediante su API MediaWiki; fallback didáctico si falla."""
    endpoint = "https://es.wikipedia.org/w/api.php"
    params = {
        "action": "query", "format": "json", "prop": "extracts",
        "exintro": 1, "explaintext": 1, "redirects": 1,
        "titles": query,
    }
    headers = {"User-Agent": "ISY0101-NotebookEducativo/1.0 (practica docente)"}
    try:
        response = requests.get(endpoint, params=params, headers=headers, timeout=12)
        response.raise_for_status()
        if "json" not in response.headers.get("Content-Type", "").lower():
            raise ValueError("El servidor no devolvió una respuesta JSON")
        payload = response.json()
        pages = payload["query"]["pages"]
        page = next(iter(pages.values()))
        if "missing" in page:
            raise LookupError(f"La entrada '{query}' no existe en Wikipedia en español")
        extract = page.get("extract", "").strip()
        if not extract:
            raise ValueError("Wikipedia no devolvió texto introductorio")
        print("Fuente: consulta real a Wikipedia en español")
        return extract[:4500]
    except (requests.RequestException, ValueError, LookupError, KeyError, StopIteration) as exc:
        print(f"Aviso: consulta Wikipedia falló ({type(exc).__name__}: {exc}).")
        if DEMO_FALLBACK and query in DEMO_TEXTS:
            print("MODO DEMOSTRACIÓN: utilizando texto local preparado; NO es una consulta a Wikipedia.")
            used_demo_data.append(query)
            return DEMO_TEXTS[query]
        raise RuntimeError(f"No fue posible consultar Wikipedia para '{query}'") from exc

def wikidata_ceo(company_name):
    """Busca el CEO en la entidad Wikidata asociada a la página de Wikipedia.

    Wikipedia (texto) y Wikidata (afirmaciones estructuradas) son fuentes distintas.
    No devuelve un nombre si el campo P169 no está disponible.
    """
    headers = {"User-Agent": "ISY0101-NotebookEducativo/1.0 (practica docente)"}
    wiki_params = {
        "action": "query", "format": "json", "prop": "pageprops",
        "ppprop": "wikibase_item", "redirects": 1, "titles": company_name,
    }
    try:
        r = requests.get("https://es.wikipedia.org/w/api.php", params=wiki_params, headers=headers, timeout=12)
        r.raise_for_status()
        pages = r.json()["query"]["pages"]
        wiki_qid = next(iter(pages.values())).get("pageprops", {}).get("wikibase_item")
        if not wiki_qid:
            raise LookupError("No existe un identificador Wikidata para la empresa")

        endpoint = "https://www.wikidata.org/w/api.php"
        r = requests.get(endpoint, params={"action": "wbgetentities", "format": "json", "ids": wiki_qid, "props": "claims"}, headers=headers, timeout=12)
        r.raise_for_status()
        entity = r.json()["entities"][wiki_qid]
        claims = entity.get("claims", {}).get("P169", [])
        # Evitar cargos explícitamente finalizados; puede haber más de un titular.
        active = [c for c in claims if not any("P582" in q for q in [c.get("qualifiers", {})])]
        if len(active) != 1:
            raise LookupError(f"Se esperó un CEO vigente inequívoco (P169); encontrados: {len(active)}")
        value = active[0]["mainsnak"]["datavalue"]["value"]
        person_qid = value["id"]
        r = requests.get(endpoint, params={"action": "wbgetentities", "format": "json", "ids": person_qid, "props": "labels", "languages": "es|en", "languagefallback": 1}, headers=headers, timeout=12)
        r.raise_for_status()
        labels = r.json()["entities"][person_qid]["labels"]
        name = (labels.get("es") or labels.get("en") or {}).get("value")
        if not name:
            raise LookupError("No se obtuvo el nombre del CEO desde Wikidata")
        print(f"Fuente: Wikidata, empresa {wiki_qid}, propiedad P169, persona {person_qid}")
        return name
    except (requests.RequestException, ValueError, LookupError, KeyError, StopIteration, TypeError) as exc:
        print(f"Aviso: consulta de CEO en Wikidata falló ({type(exc).__name__}: {exc}).")
        if DEMO_FALLBACK and company_name == "NVIDIA":
            print("MODO DEMOSTRACIÓN: CEO didáctico local (no verificado en tiempo real).")
            used_demo_data.append("CEO de NVIDIA")
            return "Jensen Huang"
        raise RuntimeError(f"No se pudo obtener el CEO de '{company_name}' desde Wikidata") from exc

def compose_report(state):
    # Se usan los textos recuperados como datos, no instrucciones.
    context = {
        "empresa": state["company"],
        "resumen_empresa": state["company_summary"],
        "ceo_extraido": state["ceo"],
        "resumen_biografia": state["bio_summary"],
    }
    message = ("Redacta un informe breve en español utilizando SOLO los datos proporcionados de Wikipedia y Wikidata. "
               "Incluye empresa, CEO identificado y breve biografía. No inventes fechas ni cargos. "
               "Aclara que la información requiere verificación de actualidad. Datos JSON:\n"
               + json.dumps(context, ensure_ascii=False))
    return llm.invoke(message).content

TOOLS = {
    "wiki_summary": wikipedia_summary,
    "wikidata_ceo": wikidata_ceo,
    "compose_report": compose_report,
}

In [8]:
def execute_plan(steps, initial_state):
    current = dict(initial_state)
    trace = []
    for step in steps:
        number, tool, output = step["id"], step["tool"], step["output"]
        try:
            argument = current if tool == "compose_report" else resolve_input(step["input"], current)
            print(f"\n[{number}] {tool} | entrada: {step['input']}")
            result = TOOLS[tool](argument)
            current[output] = result
            trace.append({"paso": number, "herramienta": tool, "estado": "OK"})
            print("Resultado:", str(result)[:350], "..." if len(str(result)) > 350 else "")
        except Exception as exc:
            trace.append({"paso": number, "herramienta": tool, "estado": "ERROR", "detalle": str(exc)})
            print(f"ERROR en paso {number}: {exc}")
            print("Se detiene la ejecución: no hay replanificación automática.")
            break
    return current, trace

final_state, execution_trace = execute_plan(steps, state)


[1] wiki_summary | entrada: {{company}}
Fuente: consulta real a Wikipedia en español
Resultado: NVIDIA Corporation es una empresa tecnológica de software y fabless que desarrolla unidades de procesamientos gráficos (GPU), interfaz de programación de aplicaciones (API) para ciencia de datos y computación de alto rendimiento, así como unidades de sistema en un chip (SoC) para la computación móvil y el mercado automotriz. NVIDIA es líder mundial ...

[2] wikidata_ceo | entrada: {{company}}
Fuente: Wikidata, empresa Q182477, propiedad P169, persona Q305177
Resultado: Jen-Hsun Huang 

[3] wiki_summary | entrada: {{ceo}}
Fuente: consulta real a Wikipedia en español
Resultado: Jensen Huang (en chino tradicional, 黃仁勳; pinyin, Huáng Rénxūn; Wade-Giles, Huang Jen-Hsün, en taiwanés N̂g Jîn-hun, nacido en Tainan, Taiwán, el 17 de febrero de 1963) es un ingeniero eléctrico y magnate tecnológico taiwanés-estadounidense. 
Es el cofundador, presidente y director ejecutivo de Nvidia, empresa especiali

## 5. Inspección de resultados

En una ejecución satisfactoria se obtiene un informe, junto con el rastro de pasos ejecutados. Si Wikidata no registra un CEO inequívoco o Wikipedia no responde, el fallo **es un resultado didáctico válido**, no una prueba de que Plan-and-Execute sea incapaz de funcionar.

In [9]:
print("TRAZA DE EJECUCIÓN")
pprint(execution_trace)
print("\nCAMPOS DEL ESTADO:", list(final_state))
if "report" in final_state:
    print("\nINFORME FINAL\n")
    print(final_state["report"])
else:
    print("\nNo se generó informe: consultar la traza para conocer la causa.")
if used_demo_data:
    print("\nADVERTENCIA: se utilizaron datos didácticos locales para:", used_demo_data)
    print("El informe NO debe presentarse como resultado verificado de Wikipedia.")


TRAZA DE EJECUCIÓN
[{'estado': 'OK', 'herramienta': 'wiki_summary', 'paso': 1},
 {'estado': 'OK', 'herramienta': 'wikidata_ceo', 'paso': 2},
 {'estado': 'OK', 'herramienta': 'wiki_summary', 'paso': 3},
 {'estado': 'OK', 'herramienta': 'compose_report', 'paso': 4}]

CAMPOS DEL ESTADO: ['company', 'company_summary', 'ceo', 'bio_summary', 'report']

INFORME FINAL

**Informe breve sobre NVIDIA Corporation y su CEO (Datos extraídos de Wikipedia y Wikidata)**

---
**Empresa: NVIDIA Corporation**
- **Descripción**: Empresa tecnológica *fabless* especializada en el desarrollo de **unidades de procesamiento gráfico (GPU)**, interfaces de programación (como **CUDA** para computación paralela y alto rendimiento), y sistemas en un chip (SoC) para mercados como **móvil, automotriz e inteligencia artificial (IA)**.
- **Líneas de negocio**:
  - GPU profesionales para sectores como **arquitectura, ingeniería, automoción, ciencia y entretenimiento**.
  - Soluciones para **supercomputación y computación

## 6. Experimento controlado: fallo de una dependencia

Simularemos que el paso 2 no logró identificar al CEO. Esto demuestra que **la ejecución siguiente no puede realizarse sin la salida anterior**. No consume tokens ni requiere Wikipedia.

In [10]:
test_state = {"company": company, "company_summary": "Texto de prueba sin información del cargo ejecutivo."}
try:
    print("Intentando ejecutar paso 3 sin 'ceo':")
    print(resolve_input("{{ceo}}", test_state))
except ValueError as exc:
    print("Dependencia detectada correctamente:", exc)

Intentando ejecutar paso 3 sin 'ceo':
Dependencia detectada correctamente: No existe un resultado utilizable para 'ceo'.


## 7. Discusión y conclusiones

**¿Qué mejora respecto al notebook 1?**
- El plan es estructurado y validado (no dependemos de dividir líneas arbitrariamente).
- Las herramientas del plan coinciden con las capacidades reales del ejecutor.
- Un paso puede utilizar datos obtenidos por pasos anteriores.
- Los errores quedan registrados y detienen la ejecución de forma controlada.

**¿Qué limitaciones se mantienen?**
- El plan es **estático**: no vuelve al LLM a replanificar.
- El ejecutor es código Python, no un segundo agente autónomo.
- No hay coordinación entre múltiples agentes, por lo tanto **esto todavía no demuestra orquestación multiagente**.
- La consulta de Wikipedia y los datos estructurados de Wikidata pueden fallar, estar incompletos o desactualizados.

**Preguntas de cierre**
1. ¿Qué cambiaría si Wikidata no identifica al CEO o proporciona una entrada ambigua?
2. ¿Cómo diseñarían un componente de replanificación para recuperarse de ese fallo?
3. ¿Qué problema nuevo aparecería si tres agentes trabajaran en paralelo con la misma información?

**Idea clave:** planificación = definir tareas y dependencias; ejecución = realizar operaciones; orquestación = coordinar los componentes/agentes y el flujo de trabajo. Un sistema con estado y control de errores no necesariamente es multiagente.